# Capstone : a service desk

Everything from the course, on 3,080 real customer-service
messages from a bank, each one already labelled, so every number in this
notebook is measured rather than asserted.

The pieces, in order : the data, tools, a guardrail, a router, the score the
router actually gets, where it goes wrong, the specialists behind it, a human
on the irreversible step, and what the whole thing would cost per month.

Runs against a local model by default.

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [1]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Optional keys. A few notebooks call a third-party API : Tavily in demos05a,
# OpenWeatherMap in demos05b, LangSmith in demos09. load_dotenv() covers the
# local path ; in Colab there is no .env, so they are read from Secrets here.
# Missing is fine, the cell that needs one says so.
if IN_COLAB:
    try:
        from google.colab import userdata
        for _name in ("TAVILY_API_KEY", "OWM_API_KEY", "LANGSMITH_API_KEY"):
            try:
                _v = userdata.get(_name)
                if _v:
                    os.environ[_name] = _v
            except Exception:
                pass
    except ImportError:
        pass


model=qwen3.5:2b


## The tickets

BANKING77 : real queries sent to a bank, labelled with 77 fine-grained intents.
Those 77 are too fine to staff a desk against, so they are grouped into five
queues. The intent is the label a human gave the message ; the queue is what we
ask the model to work out.

In [2]:
import csv, collections, random

with open(data("banking77-tickets.csv"), encoding="utf-8") as fh:
    TICKETS = list(csv.DictReader(fh))

QUEUES = sorted({t["queue"] for t in TICKETS})
print(f"{len(TICKETS)} tickets, {len({t['intent'] for t in TICKETS})} intents, "
      f"{len(QUEUES)} queues\n")

for q, n in collections.Counter(t["queue"] for t in TICKETS).most_common():
    print(f"  {q:10s} {n:5d}  {n/len(TICKETS):5.1%}")

print("\nthree real ones:")
for t in random.Random(0).sample(TICKETS, 3):
    print(f"  [{t['queue']:8s}] {t['text'][:74]}")

3080 tickets, 77 intents, 5 queues

  payments    1080  35.1%
  cards        920  29.9%
  fees         440  14.3%
  identity     360  11.7%
  fraud        280   9.1%

three real ones:
  [payments] Why was I declined at the ATM today when I was trying to make a withdraw?
  [payments] I am getting continuous failure for all my transfers. I have cross checked
  [fees    ] I would like a refund on the extra pound I was charged.


Note the shape of the problem before writing anything. The queues are not
balanced, `payments` is a third of the traffic, and a router that answered
`payments` every time would already be right 35% of the time. That is the number
to beat, and it is why accuracy alone is a weak measure.

## Customers and tools

The messages are real ; the account data behind them is invented. The agent
never sees these tables, it only ever reaches them through a tool, which is what
makes the tool call visible in the log.

In [3]:
from langchain_core.tools import tool

PLANS = ["Basic", "Plus", "Premium"]
rng = random.Random(7)
for i, t in enumerate(TICKETS):
    t["customer"] = f"C-{1000 + i % 400}"

CUSTOMERS = {f"C-{1000+i}": {"name": f"Customer {1000+i}",
                             "plan": rng.choice(PLANS),
                             "since": rng.choice([2016, 2019, 2021, 2024])}
             for i in range(400)}

TOOL_LOG = []

@tool
def lookup_customer(customer_id: str) -> str:
    """Return the plan and join year for a customer id such as C-1001."""
    TOOL_LOG.append(("lookup_customer", customer_id))
    c = CUSTOMERS.get(customer_id)
    return f"{c['name']}, {c['plan']} plan, customer since {c['since']}" if c else "unknown"

@tool
def check_sla(plan: str) -> str:
    """Return the response time promised to a plan."""
    TOOL_LOG.append(("check_sla", plan))
    return {"Premium": "2 hours", "Plus": "1 working day"}.get(plan, "3 working days")

@tool
def refund_policy(queue: str) -> str:
    """Return what this desk is allowed to refund without approval."""
    TOOL_LOG.append(("refund_policy", queue))
    return {"fees": "may refund a fee up to 25 EUR",
            "payments": "may not refund, raise a payment investigation",
            "fraud": "may not refund, must escalate to the fraud team"}.get(
            queue, "no refund authority")

print(lookup_customer.invoke({"customer_id": "C-1001"}))
print(check_sla.invoke({"plan": "Premium"}))
print(refund_policy.invoke({"queue": "fraud"}))

Customer 1001, Plus plan, customer since 2016
2 hours
may not refund, must escalate to the fraud team


## A guardrail

Real customer messages sometimes carry things that should never reach a hosted
model. This runs before anything else does.

In [4]:
import re

CARD = re.compile(r"\b(?:\d[ -]*?){13,16}\b")
IBAN = re.compile(r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,30}\b")

def screen(text):
    """Return a reason to stop, or None if the text is safe to send."""
    if CARD.search(text):
        return "looks like a card number"
    if IBAN.search(text):
        return "looks like an IBAN"
    return None

print(screen("refund my card 4111 1111 1111 1111"))
print(screen("my transfer to NL91ABNA0417164300 failed"))

flagged = [t for t in TICKETS if screen(t["text"])]
print(f"\n{len(flagged)} of {len(TICKETS)} real tickets would be held back")
for t in flagged[:3]:
    print("  ", t["text"][:78])

looks like a card number
looks like an IBAN

0 of 3080 real tickets would be held back


## The router

One model call per ticket, no tools, one word out. Everything downstream depends
on this being right, which is why it is the part we measure first.

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

llm = make_llm()

ROUTER_PROMPT = f"""You route customer messages at a bank to one desk.
Answer with exactly one word from this list and nothing else:
{", ".join(QUEUES)}

cards     lost, damaged, new, PIN, contactless, activation, delivery, ATM
payments  transfers, top-ups, refunds, declined or pending payments
fees      a charge or an exchange rate the customer disputes
fraud     unrecognised transactions, a stolen card or phone, double charges
identity  verification, personal details, closing an account, eligibility"""

def route(text):
    reply = llm.invoke([SystemMessage(ROUTER_PROMPT), HumanMessage(text)])
    answer = reply.content.strip().lower()
    for q in QUEUES:                      # tolerate a chatty model
        if q in answer:
            return q
    return "unrouted"

for t in random.Random(1).sample(TICKETS, 4):
    print(f"  {route(t['text']):10s} (labelled {t['queue']:10s}) {t['text'][:52]}")

  identity   (labelled cards     ) My PIN is blocked, what do I do?
  payments   (labelled payments  ) I would like to transfer some money from my other ba
  payments   (labelled fees      ) Will you handle EUR?


  payments   (labelled payments  ) I made a cash deposit almost a week ago but it's sti


## What the router actually scores

Four examples prove nothing. This is a stratified sample, so every queue is
represented whatever its share of the traffic, and the score is comparable
across queues.

It is roughly 60 model calls. On the classroom endpoint that is about a minute.

In [6]:
SAMPLE_PER_QUEUE = 12          # raise this to trade time for a tighter number

by_queue = collections.defaultdict(list)
for t in TICKETS:
    by_queue[t["queue"]].append(t)

sample = []
for q in QUEUES:
    sample += random.Random(3).sample(by_queue[q], SAMPLE_PER_QUEUE)

predicted = [route(t["text"]) for t in sample]
actual = [t["queue"] for t in sample]

hits = sum(p == a for p, a in zip(predicted, actual))
baseline = max(collections.Counter(t["queue"] for t in TICKETS).values()) / len(TICKETS)

print(f"routed {len(sample)} tickets")
print(f"accuracy          {hits/len(sample):.0%}  ({hits}/{len(sample)})")
print(f"always-payments   {baseline:.0%}   <- the number to beat")

routed 60 tickets
accuracy          73%  (44/60)
always-payments   35%   <- the number to beat


Accuracy on its own still hides the thing that matters. A desk cares less about
the total than about which mistakes it makes, because sending a fraud report to
the fees desk is not the same kind of error as the reverse.

## Where it goes wrong

The rows are what the ticket really was, the columns are where the router sent
it. Everything off the diagonal is a misroute.

In [7]:
matrix = collections.Counter(zip(actual, predicted))
labels = QUEUES + ["unrouted"]

print(f"{'actual \\ sent':>16s} " + " ".join(f"{q[:8]:>9s}" for q in labels))
for a in QUEUES:
    row = " ".join(f"{matrix[(a, p)]:9d}" for p in labels)
    print(f"{a:>16s} {row}")

print("\nthe misroutes, in full:")
for t, p, a in zip(sample, predicted, actual):
    if p != a:
        print(f"  {a:9s} -> {p:9s}  [{t['intent']}]  {t['text'][:56]}")

   actual \ sent     cards      fees     fraud  identity  payments  unrouted
           cards         7         0         0         4         0         1
            fees         0        12         0         0         0         0
           fraud         2         0        10         0         0         0
        identity         1         0         0         7         2         2
        payments         0         1         3         0         8         0

the misroutes, in full:
  cards     -> identity   [change_pin]  I would like to change my PIN but I am not currently in 
  cards     -> identity   [passcode_forgotten]  I don't have my access code for the app.
  cards     -> identity   [get_physical_card]  Why can't I see my PIN?
  cards     -> unrouted   [change_pin]  I want to choose a different PIN.
  cards     -> identity   [card_linking]  how do I link a card I already have?
  fraud     -> cards      [card_payment_not_recognised]  I'm not familiar with a card payment.
  fraud 

Read the misroutes rather than the number. They cluster, and the cluster is
informative : most of them are `cards` sent to `identity`, and nearly all of
those are about a PIN, a passcode or an access code. Whether "I forgot my
passcode" is a card problem or an access problem is a real question about how
the bank is organised, and the model cannot settle it.

That is worth more than a percentage. It says the taxonomy is where the work is,
and no amount of prompt engineering fixes a boundary that is not crisp.

## The specialists

Each desk gets its own agent : the same model, different instructions, different
tools and different authority. That last one is the point of splitting them.

In [8]:
from langchain.agents import create_agent

def desk(name, brief, tools):
    return create_agent(model=llm, tools=tools, system_prompt=(
        f"You work the {name} desk at a bank. {brief} "
        "Be brief. Never invent an account detail ; use a tool or say you cannot."))

DESKS = {
  "cards":    desk("cards", "Handle cards, PINs and deliveries.",
                   [lookup_customer, check_sla]),
  "payments": desk("payments", "Handle transfers, top-ups and refunds. "
                   "You may not issue a refund yourself.",
                   [lookup_customer, check_sla, refund_policy]),
  "fees":     desk("fees", "Handle disputed charges and exchange rates. "
                   "You may refund a fee up to 25 EUR.",
                   [lookup_customer, refund_policy]),
  "identity": desk("identity", "Handle verification and personal details. "
                   "Never confirm identity yourself.", [lookup_customer]),
  "fraud":    desk("fraud", "Handle suspected fraud. Every case is escalated "
                   "to a human, without exception.", [lookup_customer, refund_policy]),
}
print("desks:", ", ".join(DESKS))

desks: cards, payments, fees, identity, fraud


## Putting it together

Guardrail, router, desk, and a hard stop on anything the fraud desk touches.

In [9]:
def handle(ticket):
    blocked = screen(ticket["text"])
    if blocked:
        return {"queue": "held", "reply": f"held back : {blocked}", "escalated": False}

    queue = route(ticket["text"])
    if queue not in DESKS:
        return {"queue": queue, "reply": "no desk for this", "escalated": True}

    TOOL_LOG.clear()
    result = DESKS[queue].invoke({"messages": [HumanMessage(
        f"Customer {ticket['customer']} writes: {ticket['text']}")]})
    return {"queue": queue,
            "reply": result["messages"][-1].content.strip(),
            "tools": list(TOOL_LOG),
            "escalated": queue == "fraud"}

for t in random.Random(11).sample(TICKETS, 3):
    r = handle(t)
    print(f"--- [{t['queue']} -> {r['queue']}] {t['text'][:60]}")
    print(f"    tools: {r.get('tools')}")
    print(f"    {r['reply'][:150]}\n")

--- [payments -> payments] My transfer is still coming up as pending.
    tools: [('lookup_customer', 'C-1252')]
    I'll check the status of your transfer.



--- [payments -> payments] I transferred my funds, why did it not go through?
    tools: [('refund_policy', 'C-1292')]
    I cannot issue a refund myself. I need to check the transfer details and verify if there was an error with your account or funds. Please provide the t



--- [identity -> identity] Please delete my account.
    tools: [('lookup_customer', 'C-1307')]
    I cannot delete your account. I do not have the authority to remove accounts or perform actions that affect your personal details. Please contact your



## Stopping for a human

The fraud desk never completes on its own. The agent prepares the case, a person
decides, and the decision is what gets recorded.

In [10]:
fraud_ticket = next(t for t in TICKETS if t["queue"] == "fraud")

def with_approval(ticket, approve):
    r = handle(ticket)
    if not r["escalated"]:
        return r["reply"]
    if not approve:
        return f"[held for a human] {r['reply'][:110]}"
    return f"[approved by a human] {r['reply'][:110]}"

print(fraud_ticket["text"], "\n")
print(with_approval(fraud_ticket, approve=False), "\n")
print(with_approval(fraud_ticket, approve=True))

My entire gym bag, including my wallet, was stolen out of my locker today. Everything in my wallet is gone - how do I block the card to make it can't be used? 



[held for a human] I cannot block the card or provide a refund. This is a suspected fraud case and will be escalated to a human a 



[approved by a human] I cannot block the card or provide a refund. This is a suspected fraud case and will be escalated to a human a


## What it would cost

The desk in question handles ten thousand tickets a month. Two model calls per
ticket, the router and the desk.

In [11]:
from langchain_core.callbacks import get_usage_metadata_callback

probe = random.Random(5).sample(TICKETS, 5)
with get_usage_metadata_callback() as cb:
    for t in probe:
        handle(t)
    usage = list(cb.usage_metadata.values())[0]

per_ticket_in = usage["input_tokens"] / len(probe)
per_ticket_out = usage["output_tokens"] / len(probe)
print(f"per ticket : {per_ticket_in:.0f} in, {per_ticket_out:.0f} out")

for name, pin, pout in [("small model", 0.50, 1.50), ("large model", 3.00, 15.00)]:
    monthly = (per_ticket_in * 10_000 / 1e6) * pin + (per_ticket_out * 10_000 / 1e6) * pout
    print(f"  {name:12s} {monthly:8.2f} EUR per month at 10,000 tickets")

per ticket : 1068 in, 61 out
  small model      6.25 EUR per month at 10,000 tickets
  large model     41.14 EUR per month at 10,000 tickets


Compare that with the misroute rate. A desk that is right 80% of the time and
costs 40 EUR a month is a different proposition from one that is right 95% of
the time and costs 500, and neither number means anything without the other.

## Things to try

Nothing here is required.

1. Raise `SAMPLE_PER_QUEUE` to 40 and see how much the accuracy moves. If it
   barely moves, the first sample was already big enough.
2. Give the router three worked examples in its prompt and re-score it. This is
   the cheapest change on the list, and usually the largest.
3. Collapse `fees` into `payments` and re-score. Most misroutes are on that
   boundary, so the accuracy should jump, and the desk becomes less useful.
4. Route on the 77 intents instead of the 5 queues and watch it fall apart.
5. Send the full ticket text to one desk and the first sentence to another.
   Cheaper, and usually no worse.